In [2]:
from langchain.agents import create_agent
import os
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig
from langchain_core.tools import tool
from pathlib import Path

C:\Users\kasia\AppData\Local\Temp\ipykernel_3712\866319617.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


In [6]:
# path topdf files
pdf_folder = "../data/"
pdf_files = [
    "recipes1.pdf"]
pages=[]

# load pdf files
for pdf_file in pdf_files:
    pdf_path = os.path.join(pdf_folder, pdf_file)

    loader = PyMuPDFLoader(file_path=pdf_path)
    pdf_pages = loader.load()

    print(f"Loaded {pdf_file}: {len(pdf_pages)} pages")

    pages.extend(pdf_pages)

# splitting the documents into smaller chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=100,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)

docs = text_splitter.split_documents(pages)
print("Number of loaded pages:", len(pages))
print("Number of created fragments:", len(docs))



Loaded recipes1.pdf: 92 pages
Number of loaded pages: 92
Number of created fragments: 159


In [6]:
for idx,doc in enumerate(docs[75:79]):
  print(f"[{idx}].{doc}\n\n")


[0].page_content='Nutrition Facts Per Serving: Calories: 205 | Total Fat: 7 g | Saturated Fat: 1 g   
Sodium: 270 mg | Total Carbohydrate: 15 g | Protein: 23 g 
 
For more recipes, please visit www.nutrition.va.gov' metadata={'producer': '', 'creator': '', 'creationdate': '2022-04-05T07:55:20-07:00', 'source': '/content/drive/MyDrive/data/recipes/recipes1.pdf', 'file_path': '/content/drive/MyDrive/data/recipes/recipes1.pdf', 'total_pages': 92, 'format': 'PDF 1.7', 'title': '2022 Cooking Around the World Cookbook', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-04-24T15:11:00-04:00', 'trapped': '', 'modDate': "D:20260424151100-04'00'", 'creationDate': "D:20220405075520-07'00'", 'page': 43}


[1].page_content='40 
 
 
 Table of Contents 
         
Yellow Split Pea Soup 
Prep: 10 minutes | Cook: 2 hours  | Total:  2 hours 10 minutes 
Yield: 4 servings | Serving Size: ~1 cup  
Region: North America | Country: Canada 
Ingredients 
1 tablespoon unsalted butter  
1 medium carro

In [7]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

c:\Users\kasia\anaconda3\envs\rag-agent\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\kasia\anaconda3\envs\rag-agent\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\kasia\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. 

In [8]:
db_path = "../data/chroma_db"

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory=db_path
    )

In [9]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 6}
    )

In [10]:
similar = retriever.invoke('What can we cook with potatoes?')
for doc in similar:
  print(doc.page_content[:300] + "...\n---")

69 
 
 
 Table of Contents 
         
Vegetable-Loaded Potato Salad 
Prep: 10 minutes | Cook: 20 minutes | Total: 30 minutes  
Yield: 4 servings | Serving Size: ~1 cup 
Region: Mediterranean Europe | Country: Portugal 
Ingredients 
Water 
1 teaspoon plus 1 pinch salt, divided 
2 medium (5- to 7-ounc...
---
61 
 
 
 Table of Contents 
         
Skillet Shepherd’s Pie (Cottage Pie) 
Prep: 10 minutes | Cook: 20 minutes | Total: 30 minutes  
Yield: 3 servings | Serving Size: 1½ cups 
Region: Western Europe | Country: Ireland 
Ingredients 
½ pound yellow or red potatoes (about  
2 medium potatoes or 4 smal...
---
63 
 
 
 Table of Contents 
         
Lemon Baked Fish and Chips 
Prep: 10 minutes | Cook: 15 minutes | Total: 25 minutes  
Yield: 2 servings | Serving Size: 1 (~5-ounce) fish fillet with half of the fries 
Region: Western Europe | Country: United Kingdom 
Ingredients 
1 large (10- to 12-ounce) Russ...
---
55 
 
 
 Table of Contents 
         
Sweet Potato Latkes (Potato Fritters) 

In [15]:
from dotenv import load_dotenv
load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")

if not api_key:
    raise ValueError("No Google API key found. Please set the GOOGLE_API_KEY environment variable.")

In [16]:
@tool
def search_recipes(query: str) -> str:
    """
    Search the PDF knowledge base for recipe details.
    Use this for any questions about recipes, ingredients, cooking instructions, or preparation time.
    Answer based strictly only on the PDF documents. Always provide the source of the information.
    """
    print("DEBUG: search_recipes used")
    docs = retriever.invoke(query)
    if not docs:
        return "No relevant information was found in the recipe knowledge base."

    results = []

    for doc in docs:
        source = Path(doc.metadata["source"]).name
        page = doc.metadata.get("page", 0) + 1

        results.append(
            f"Source: {source}, page: {page}\n"
            f"{doc.page_content}"
        )

    return "\n\n---\n\n".join(results)

@tool
def ingredient_converter(
    quantity: float,
    original_servings: int,
    new_servings: int
) -> float:
    """
    Calculate a new ingredient quantity when changing the number of servings.
    Use this tool whenever the user asks to scale an ingredient.
    Use this tool for all quantities, including item counts.
    You can only do the calculations.
    """

    print("DEBUG: ingredient_converter used")

    if original_servings <= 0 or new_servings <= 0:
        raise ValueError("Number of servings must be greater than zero.")

    return quantity / original_servings * new_servings

print(
    search_recipes.invoke({
        "query": "recipe with potatoes"
    })
)

result = ingredient_converter.invoke({
    "quantity": 200,
    "original_servings": 4,
    "new_servings": 6
})

print("New quantity:", result)
tools = [search_recipes, ingredient_converter]

DEBUG: search_recipes used
Source: recipes1.pdf, page: 74
69 
 
 
 Table of Contents 
         
Vegetable-Loaded Potato Salad 
Prep: 10 minutes | Cook: 20 minutes | Total: 30 minutes  
Yield: 4 servings | Serving Size: ~1 cup 
Region: Mediterranean Europe | Country: Portugal 
Ingredients 
Water 
1 teaspoon plus 1 pinch salt, divided 
2 medium (5- to 7-ounce) red or Yukon gold 
potatoes, cut into to ½-inch chunks 
2 medium carrots, peeled, sliced ½-inch-thick 
½ medium onion, sliced ½-inch-thick 
2 eggs, in shell 
1 cup chopped green beans, frozen or fresh 
3 tablespoons red wine vinegar 
2 tablespoons olive oil 
1 teaspoon Dijon mustard 
¼ teaspoon paprika 
1 pinch ground white or black pepper 
2 tablespoons chopped fresh parsley 
1 clove garlic, minced (about ½ teaspoon) 
1 tablespoon capers, drained 
Directions 
1. Fill a large saucepan or stockpot about two-thirds full with 
water. Add the potatoes, carrots, onion, and 1 teaspoon of 
the salt. (Potatoes absorb ~7-10% of salt from co

In [17]:
def parse_result(res):
  to_parse = res["messages"][-1]
  return to_parse.content[0]['text']


In [19]:
agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=tools,
    system_prompt="""
You are a helpful culinary assistant on a desert island. Survivors of the Titanic disaster need to be fed so always refer to people as "survivors".
Answer questions strictly based on the provided PDF recipe base and available tools.
Always use 'search_recipes' for questions about recipes or ingredients. When the user asks for a recipe, provide the complete recipe found in the PDF.
If the recipe is incomplete do not include it in your answer.
If information is missing from the documents, clearly state that it was not found.
Use 'ingredient_converter' whenever ingredient amounts need to be adjusted for a different number of servings.
""",
    checkpointer=InMemorySaver()
    )

config: RunnableConfig = {"configurable": {"thread_id": "1"}}

def run_agent(query: str, config: RunnableConfig):
  result = agent.invoke({"messages": [{"role": "user", "content": query}]}, config)
  return parse_result(result)

result_list = list()

result_list.append(run_agent("Give me a recipe that includes potatoes", config))
result_list.append(run_agent("How many ingredients are needed for 8 servings according to the 'Vegetable-Loaded Potato Salad' recipe?", config))
result_list.append(run_agent("I have 4 strawberries, what can I do with them", config))
result_list.append(run_agent("What day of the week is today?", config))
result_list.append(run_agent("What did I ask you in the beginning?", config))

for res in result_list:
  print(res)


DEBUG: search_recipes used
DEBUG: ingredient_converter used
DEBUG: ingredient_converter used
DEBUG: ingredient_converter used
DEBUG: ingredient_converter used
DEBUG: ingredient_converter used
DEBUG: ingredient_converter used
DEBUG: ingredient_converter used
DEBUG: ingredient_converter used
DEBUG: ingredient_converter used
DEBUG: ingredient_converter used
DEBUG: ingredient_converter used
DEBUG: ingredient_converter used
DEBUG: search_recipes used
Here are some recipes containing potatoes that the survivors can prepare:

### Vegetable-Loaded Potato Salad (Portugal)
*Source: recipes1.pdf, page 74*

**Ingredients**
* Water
* 1 teaspoon plus 1 pinch salt, divided
* 2 medium (5- to 7-ounce) red or Yukon gold potatoes, cut into ½-inch chunks
* 2 medium carrots, peeled, sliced ½-inch-thick
* ½ medium onion, sliced ½-inch-thick
* 2 eggs, in shell
* 1 cup chopped green beans, frozen or fresh
* 3 tablespoons red wine vinegar
* 2 tablespoons olive oil
* 1 teaspoon Dijon mustard
* ¼ teaspoon paprik

Tests show that the agent is able to decide when to use the tools and when to answer directly. It can also handle questions that are not related to recipes, but it will always refer to people as "survivors" and will not provide information that is not found in the PDF documents.